# 24c — Yorkshire SDP Case Study Validation v1

This notebook focuses on Yorkshire and neighbouring SDP case-study areas to test the model against actual SDP campaign evidence.

Core questions:

- What do SDP-contested Yorkshire wards look like demographically?
- Which tribes appear in high-performing SDP wards?
- Do successful SDP cases resemble North West watchlist wards?
- Does Middleton Park / Leeds / South Yorkshire evidence support or challenge the current tribe interpretation?

Outputs are written to:

```text
data/processed/yorkshire_case_study_v1/
```

In [2]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_DIRS = [
    PROCESSED_DIR / "sdp_validation_v2",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "yorkshire_case_study_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SDP_PROFILE_FILENAME = "sdp_campaign_wards_profile_v2.csv"
NW_REVIEW_FILENAME = "north_west_revised_consolidated_review_v2.csv"

YORKSHIRE_REGION_NAMES = ["Yorkshire and The Humber", "Yorkshire", "Yorkshire & The Humber"]
KEY_CASE_STUDY_LADS = [
    "Leeds", "Barnsley", "Doncaster", "Rotherham", "Sheffield", "Wakefield",
    "Kirklees", "Calderdale", "Bradford", "York", "North Yorkshire", "East Riding of Yorkshire", "Kingston upon Hull, City of"
]
SOUTH_YORKSHIRE_LADS = ["Barnsley", "Doncaster", "Rotherham", "Sheffield"]
KEY_WARD_NAMES = ["Middleton Park", "Dearne South", "Wath", "Beeston and Holbeck", "Firth Park"]

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\yorkshire_case_study_v1


In [3]:
def find_file(filename, required=True):
    for folder in INPUT_DIRS + [Path("/mnt/data")]:
        p = folder / filename
        if p.exists():
            return p
    for root in [PROCESSED_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(filename)
    return None


def clean_text(x):
    if pd.isna(x): return ""
    import re
    x = str(x).lower().strip().replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def read_csv(filename, required=True):
    p = find_file(filename, required=required)
    if p is None:
        return None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df

## 24c.1 Load SDP validation file and identify Yorkshire case-study rows

In [4]:
sdp = read_csv(SDP_PROFILE_FILENAME)
sdp["analysis_region"] = sdp.get("analysis_region", pd.Series("", index=sdp.index)).fillna("")
sdp["LAD25NM"] = sdp.get("LAD25NM", pd.Series("", index=sdp.index)).fillna(sdp.get("council_name", ""))
sdp["WD25NM"] = sdp.get("WD25NM", pd.Series("", index=sdp.index)).fillna(sdp.get("ward_name", ""))

sdp["clean_lad"] = sdp["LAD25NM"].map(clean_text)
sdp["clean_ward"] = sdp["WD25NM"].map(clean_text)

case_lads_clean = {clean_text(x) for x in KEY_CASE_STUDY_LADS}
south_yorks_clean = {clean_text(x) for x in SOUTH_YORKSHIRE_LADS}
key_wards_clean = {clean_text(x) for x in KEY_WARD_NAMES}

yorkshire_mask = (
    sdp["analysis_region"].isin(YORKSHIRE_REGION_NAMES)
    | sdp["clean_lad"].isin(case_lads_clean)
    | sdp["clean_ward"].isin(key_wards_clean)
)

yorks = sdp[yorkshire_mask].copy()

# Case-study labels.
def case_area(row):
    lad = row["clean_lad"]
    ward = row["clean_ward"]
    if "middleton park" in ward:
        return "Leeds - Middleton Park"
    if "beeston" in ward and "holbeck" in ward:
        return "Leeds - Beeston and Holbeck"
    if lad == clean_text("Leeds"):
        return "Leeds - Other"
    if lad == clean_text("Barnsley"):
        return "Barnsley"
    if lad == clean_text("Doncaster"):
        return "Doncaster"
    if lad == clean_text("Rotherham"):
        return "Rotherham"
    if lad == clean_text("Sheffield"):
        return "Sheffield"
    if lad in south_yorks_clean:
        return "South Yorkshire - Other"
    return "Other Yorkshire / Humber"

yorks["case_study_area"] = yorks.apply(case_area, axis=1)

print("Yorkshire / case-study SDP rows:", len(yorks))
display(yorks["case_study_area"].value_counts(dropna=False).reset_index(name="rows"))

Loaded sdp_campaign_wards_profile_v2.csv: (11, 57) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_validation_v2\sdp_campaign_wards_profile_v2.csv


ValueError: Cannot set a DataFrame with multiple columns to the single column case_study_area

## 24c.2 Produce case-study summaries

In [ ]:
def summarise(group_cols, filename):
    cols = [c for c in group_cols if c in yorks.columns]
    if not cols:
        return pd.DataFrame()
    out = (
        yorks.groupby(cols, dropna=False, as_index=False)
        .agg(
            contests=("candidate_name", "size"),
            years=("election_year", lambda s: "; ".join(map(str, sorted(set(s.dropna().astype(int)))) if s.notna().any() else "")),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
            median_sdp_vote_share=("sdp_vote_share_effective", "median"),
            max_sdp_vote_share=("sdp_vote_share_effective", "max"),
            mean_model_score=("initial_watchlist_score", "mean"),
            max_model_score=("initial_watchlist_score", "max"),
        )
        .sort_values(["max_sdp_vote_share", "total_sdp_votes"], ascending=False)
    )
    out.to_csv(OUTPUT_DIR / filename, index=False)
    return out

case_wards = summarise(["case_study_area", "LAD25NM", "WD25NM", "WD25CD", "dominant_cluster_name", "second_cluster_name", "latest_election_top_party_bucket"], "yorkshire_sdp_case_study_wards_v1.csv")
by_tribe = summarise(["dominant_cluster_name"], "yorkshire_sdp_performance_by_tribe_v1.csv")
by_party = summarise(["latest_election_top_party_bucket"], "yorkshire_sdp_performance_by_latest_party_v1.csv")
by_area = summarise(["case_study_area"], "yorkshire_sdp_performance_by_case_area_v1.csv")

# Specific Middleton Park profile.
middleton = yorks[yorks["case_study_area"].eq("Leeds - Middleton Park")].copy()
middleton.to_csv(OUTPUT_DIR / "middleton_park_case_study_profile_v1.csv", index=False)

# High performance cases.
high_perf = yorks.sort_values(["sdp_vote_share_effective", "sdp_votes"], ascending=False).head(50).copy()
high_perf.to_csv(OUTPUT_DIR / "yorkshire_high_performance_sdp_cases_v1.csv", index=False)

print("Case ward summary:")
display(case_wards.head(20))
print("Performance by tribe:")
display(by_tribe)
print("Middleton Park rows:", len(middleton))
display(middleton.head(10))

## 24c.3 Compare Yorkshire SDP evidence with North West watchlist profile

This comparison is indicative only. It checks whether high-performing SDP wards resemble the current North West model's preferred lanes and tribes.

In [ ]:
# Load NW revised review if available for broad comparison.
nw = read_csv(NW_REVIEW_FILENAME, required=False)
comparison_rows = []

if nw is not None:
    # Tribe comparison.
    y_tribe = yorks["dominant_cluster_name"].value_counts(normalize=True).rename("yorkshire_sdp_share")
    nw_tribe = nw["dominant_cluster_name"].value_counts(normalize=True).rename("north_west_review_share")
    tribe_compare = pd.concat([y_tribe, nw_tribe], axis=1).fillna(0).reset_index().rename(columns={"index": "dominant_cluster_name"})
    tribe_compare["difference_yorkshire_minus_nw"] = tribe_compare["yorkshire_sdp_share"] - tribe_compare["north_west_review_share"]
    tribe_compare.to_csv(OUTPUT_DIR / "yorkshire_vs_north_west_tribe_comparison_v1.csv", index=False)
    display(tribe_compare.sort_values("difference_yorkshire_minus_nw", ascending=False))

    # Latest top party comparison.
    if "latest_election_top_party_bucket" in yorks.columns and "latest_election_top_party_bucket" in nw.columns:
        y_party = yorks["latest_election_top_party_bucket"].value_counts(normalize=True).rename("yorkshire_sdp_share")
        nw_party = nw["latest_election_top_party_bucket"].value_counts(normalize=True).rename("north_west_review_share")
        party_compare = pd.concat([y_party, nw_party], axis=1).fillna(0).reset_index().rename(columns={"index": "latest_election_top_party_bucket"})
        party_compare["difference_yorkshire_minus_nw"] = party_compare["yorkshire_sdp_share"] - party_compare["north_west_review_share"]
        party_compare.to_csv(OUTPUT_DIR / "yorkshire_vs_north_west_latest_party_comparison_v1.csv", index=False)
        display(party_compare.sort_values("difference_yorkshire_minus_nw", ascending=False))

# Case-study headline summary.
summary = pd.DataFrame([{
    "case_study_rows": len(yorks),
    "matched_to_model_rows": int(yorks.get("matched_to_model_v2", pd.Series(False)).fillna(False).astype(bool).sum()) if "matched_to_model_v2" in yorks.columns else int(yorks["initial_watchlist_score"].notna().sum()),
    "middleton_park_rows": len(middleton),
    "max_sdp_vote_share": yorks["sdp_vote_share_effective"].max(),
    "mean_sdp_vote_share": yorks["sdp_vote_share_effective"].mean(),
    "top_dominant_tribe": yorks["dominant_cluster_name"].mode().iloc[0] if yorks["dominant_cluster_name"].notna().any() else "unknown",
    "top_latest_party": yorks["latest_election_top_party_bucket"].mode().iloc[0] if yorks["latest_election_top_party_bucket"].notna().any() else "unknown",
}])
summary.to_csv(OUTPUT_DIR / "yorkshire_case_study_summary_v1.csv", index=False)
display(summary)